# HW2 Colab Starter — ConvNeXt Ablation + FCOS vs RetinaNet
**CS-GY 6953 Deep Learning — Spring 2026**

*Compute Diary:* Run the next cell at the **start and end** of each session.  
Keep total GPU time ≤ **90 minutes** across both problems.

In [ ]:
from datetime import datetime
import os, json
stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
os.makedirs("diary", exist_ok=True)
with open("diary/compute_diary.txt", "a") as f:
    f.write(json.dumps({"event": "timestamp", "when": stamp}) + "\n")
print("⏱️ Compute diary timestamp:", stamp)

## 0) Environment & GPU Check

In [ ]:
import torch, platform, torchvision
print("Python    :", platform.python_version())
print("Torch     :", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Install torchgeo for Problem 4 (NWPU VHR-10)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchgeo", "pycocotools"])
print("✅ torchgeo and pycocotools installed")

## 1) Reproducibility — Per-Student Seed

**⚠️ Replace `YOUR_NETID_HERE` with your actual NYU NetID (e.g. `abc1234`).  
Every metric you report is unique to your seed — do not change it after starting experiments.**

In [ ]:
import hashlib, random, numpy as np, torch

NETID = "YOUR_NETID_HERE"   # <-- change this
SEED  = int(hashlib.sha256(NETID.encode()).hexdigest(), 16) % 10000
print(f"NetID: {NETID}  |  Seed: {SEED}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)
print("✅ Seed set to", SEED)

---
# PROBLEM 3: Deconstructing a Modern CNN — ConvNeXt Ablation (25 pts)

We fine-tune **ConvNeXt-Tiny** on **PatchCamelyon** (binary histopathology classification) and systematically remove one modern design choice at a time to measure each component's contribution.

## P3 — 1) Data: PatchCamelyon (with Colab-safe fallback mode)

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np, gc, os

# ── Data mode switch (prevents Colab runtime crashes on huge PCAM download/open)
# Default is "light" for Colab stability; set P3_DATA_MODE=pcam to force full PCAM.
P3_DATA_MODE    = os.environ.get("P3_DATA_MODE", "light").strip().lower()
SUBSET_FRACTION = 0.05
VAL_CAP         = 4096
BATCH_SIZE_P3   = 32
ROOT_P3         = "./data_pcam"
ROOT_LIGHT_P3   = "./data_cifar10"

if P3_DATA_MODE == "pcam":
    tfm_train_p3 = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.7008, 0.5384, 0.6916],
                             std =[0.2350, 0.2774, 0.2128]),
    ])
    tfm_val_p3 = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.7008, 0.5384, 0.6916],
                             std =[0.2350, 0.2774, 0.2128]),
    ])

    print("⬇️  Downloading / opening PCAM train split...")
    pcam_train_full = datasets.PCAM(ROOT_P3, split="train", download=True, transform=tfm_train_p3)
    print("⬇️  Downloading / opening PCAM val split...")
    pcam_val_full = datasets.PCAM(ROOT_P3, split="val", download=True, transform=tfm_val_p3)

    rng = np.random.default_rng(SEED)
    subset_size = int(len(pcam_train_full) * SUBSET_FRACTION)
    train_idx = rng.choice(len(pcam_train_full), size=subset_size, replace=False)
    val_idx = rng.choice(len(pcam_val_full), size=min(VAL_CAP, len(pcam_val_full)), replace=False)

    pcam_train = Subset(pcam_train_full, train_idx)
    pcam_val = Subset(pcam_val_full, val_idx)

    train_loader_p3 = DataLoader(pcam_train, batch_size=BATCH_SIZE_P3, shuffle=True,
                                 num_workers=0, pin_memory=False)
    val_loader_p3 = DataLoader(pcam_val, batch_size=BATCH_SIZE_P3, shuffle=False,
                               num_workers=0, pin_memory=False)

    print(f"[P3] mode=pcam | train subset={len(pcam_train):,} | val subset={len(pcam_val):,}")

else:
    # Lightweight proxy mode: binary CIFAR10 (cat vs dog) to keep notebook runnable.
    tfm_train_p3 = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                             std=[0.2470, 0.2435, 0.2616]),
    ])
    tfm_val_p3 = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                             std=[0.2470, 0.2435, 0.2616]),
    ])

    cifar_train_full = datasets.CIFAR10(ROOT_LIGHT_P3, train=True, download=True, transform=tfm_train_p3)
    cifar_val_full = datasets.CIFAR10(ROOT_LIGHT_P3, train=False, download=True, transform=tfm_val_p3)

    cat_id, dog_id = 3, 5
    train_idx = [i for i, y in enumerate(cifar_train_full.targets) if y in (cat_id, dog_id)]
    val_idx = [i for i, y in enumerate(cifar_val_full.targets) if y in (cat_id, dog_id)]

    rng = np.random.default_rng(SEED)
    train_idx = rng.choice(train_idx, size=min(8000, len(train_idx)), replace=False)
    val_idx = rng.choice(val_idx, size=min(2000, len(val_idx)), replace=False)

    class BinarySubset(torch.utils.data.Dataset):
        def __init__(self, base, indices):
            self.base = base
            self.indices = list(indices)
        def __len__(self):
            return len(self.indices)
        def __getitem__(self, i):
            x, y = self.base[self.indices[i]]
            y = 0 if y == cat_id else 1
            return x, torch.tensor(y, dtype=torch.long)

    pcam_train = BinarySubset(cifar_train_full, train_idx)
    pcam_val = BinarySubset(cifar_val_full, val_idx)

    train_loader_p3 = DataLoader(pcam_train, batch_size=BATCH_SIZE_P3, shuffle=True,
                                 num_workers=0, pin_memory=False)
    val_loader_p3 = DataLoader(pcam_val, batch_size=BATCH_SIZE_P3, shuffle=False,
                               num_workers=0, pin_memory=False)

    print(f"[P3] mode=light (binary CIFAR10 cat-vs-dog) | train={len(pcam_train):,} | val={len(pcam_val):,}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Dataloaders ready")

## P3 — 2) Baseline ConvNeXt-Tiny Fine-Tuning

In [ ]:
import torch.nn as nn
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

EPOCHS_P3 = 3
LR_P3     = 2e-4
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP   = torch.cuda.is_available()

def make_convnext_baseline():
    """Load ConvNeXt-Tiny with ImageNet pretrained weights, adapt classifier head for 2 classes."""
    model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, 2)
    return model.to(device)

def train_model_p3(model, label="baseline"):
    """Train model for EPOCHS_P3 epochs; return history dict."""
    set_seed(SEED)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_P3, weight_decay=0.05)
    criterion = nn.CrossEntropyLoss()
    scaler    = torch.amp.GradScaler(enabled=USE_AMP)
    history   = {"train_loss": [], "val_acc": []}

    for epoch in range(1, EPOCHS_P3 + 1):
        model.train()
        running = 0.0
        for x, y in train_loader_p3:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * x.size(0)
        train_loss = running / len(train_loader_p3.dataset)

        model.eval()
        correct = total = 0
        with torch.no_grad(), torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
            for x, y in val_loader_p3:
                x, y  = x.to(device), y.to(device)
                preds = model(x).argmax(1)
                correct += (preds == y).sum().item()
                total   += y.size(0)
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["val_acc"].append(val_acc)
        print(f"[{label}] Epoch {epoch:02d} | loss={train_loss:.4f} | val_acc={val_acc:.4f}")

    return history

print("✅ Training utilities ready (AMP enabled).")

In [ ]:
# Run baseline
baseline_model   = make_convnext_baseline()
results_p3       = {}
results_p3["baseline"] = train_model_p3(baseline_model, label="baseline")

## P3 — 3) Ablation Functions

Each function below should return a **modified ConvNeXt-Tiny** with exactly one change applied.  
Fill in every `# YOUR CODE HERE` block.

In [ ]:
import copy

# ─────────────────────────────────────────────────────────────────────────────
# Ablation 1: Replace all LayerNorm with BatchNorm2d
# ─────────────────────────────────────────────────────────────────────────────
def ablate_layernorm_to_batchnorm(base_model):
    """
    Return a deep copy of base_model with every nn.LayerNorm replaced by
    a BatchNorm2d-based module.
    IMPORTANT: ConvNeXt uses both NCHW and NHWC layouts internally, so a
    plain nn.BatchNorm2d replacement can break due to channel-axis mismatch.
    Implement a layout-safe wrapper (or equivalent) and use normalized_shape[-1].
    Hint: iterate model.modules() or use a recursive replacement helper.
    """
    model = copy.deepcopy(base_model)
    # YOUR CODE HERE
    raise NotImplementedError
    return model.to(device)


# ─────────────────────────────────────────────────────────────────────────────
# Ablation 2: Replace all GELU activations with ReLU
# ─────────────────────────────────────────────────────────────────────────────
def ablate_gelu_to_relu(base_model):
    """
    Return a deep copy of base_model with every nn.GELU replaced by nn.ReLU.
    """
    model = copy.deepcopy(base_model)
    # YOUR CODE HERE
    raise NotImplementedError
    return model.to(device)


# ─────────────────────────────────────────────────────────────────────────────
# Ablation 3: Replace 7×7 depthwise conv with 3×3 standard conv
# ─────────────────────────────────────────────────────────────────────────────
def ablate_7x7_to_3x3(base_model):
    """
    Return a deep copy of base_model where each depthwise Conv2d with
    kernel_size=7 is replaced with a standard (groups=1) Conv2d with
    kernel_size=3 and padding=1.
    Keep in_channels == out_channels and set groups=1.
    Re-initialize the new conv weights with kaiming_normal_.
    """
    model = copy.deepcopy(base_model)
    # YOUR CODE HERE
    raise NotImplementedError
    return model.to(device)


# ─────────────────────────────────────────────────────────────────────────────
# Ablation 4: Inverted bottleneck → standard bottleneck
# ─────────────────────────────────────────────────────────────────────────────
def ablate_inverted_to_standard_bottleneck(base_model):
    """
    ConvNeXt-Tiny's CNBlock expands channels by 4× in the pointwise conv.
    Replace the two pointwise Conv2d layers (kernel=1) in each CNBlock so
    that the hidden dimension is channels // 4 instead of channels * 4.
    Adjust both the expansion conv and the projection conv accordingly.
    Re-initialize new weights with kaiming_normal_.

    Hint: inspect torchvision.models.convnext.CNBlock for the layer names.
    """
    model = copy.deepcopy(base_model)
    # YOUR CODE HERE
    raise NotImplementedError
    return model.to(device)


# ─────────────────────────────────────────────────────────────────────────────
# Ablation 5: Remove Stochastic Depth (drop path)
# ─────────────────────────────────────────────────────────────────────────────
def ablate_remove_stochastic_depth(base_model):
    """
    Return a deep copy of base_model where every torchvision StochasticDepth
    layer (torchvision.ops.StochasticDepth) is replaced by nn.Identity().
    """
    from torchvision.ops import StochasticDepth
    model = copy.deepcopy(base_model)
    # YOUR CODE HERE
    raise NotImplementedError
    return model.to(device)


print("✅ Ablation function stubs loaded.")

## P3 — 4) Run All Ablations

In [ ]:
import gc

ablation_fns = {
    "abl1_LN_to_BN"        : ablate_layernorm_to_batchnorm,
    "abl2_GELU_to_ReLU"    : ablate_gelu_to_relu,
    "abl3_7x7_to_3x3"      : ablate_7x7_to_3x3,
    "abl4_inv_to_std_btlnk": ablate_inverted_to_standard_bottleneck,
    "abl5_no_stoch_depth"  : ablate_remove_stochastic_depth,
}

for name, fn in ablation_fns.items():
    print(f"\n{'='*60}")
    print(f"Running ablation: {name}")
    ablated_model = fn(make_convnext_baseline())
    results_p3[name] = train_model_p3(ablated_model, label=name)
    # Free the ablated model immediately to recover GPU + CPU RAM
    del ablated_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n✅ All ablations complete.")

## P3 — 5) Results Table & Contribution Ranking (Part A + Part B)

In [ ]:
import pandas as pd

baseline_acc = results_p3["baseline"]["val_acc"][-1]

rows = []
label_map = {
    "baseline"            : "Baseline (ConvNeXt-Tiny)",
    "abl1_LN_to_BN"       : "Ablation 1: LN → BN",
    "abl2_GELU_to_ReLU"   : "Ablation 2: GELU → ReLU",
    "abl3_7x7_to_3x3"     : "Ablation 3: 7×7 → 3×3",
    "abl4_inv_to_std_btlnk": "Ablation 4: Inv → Std Bottleneck",
    "abl5_no_stoch_depth"  : "Ablation 5: No Stochastic Depth",
}

for key, label in label_map.items():
    acc  = results_p3[key]["val_acc"][-1]
    drop = baseline_acc - acc if key != "baseline" else float("nan")
    rows.append({"Model": label, "Val Acc (%)": round(acc * 100, 2),
                 "Acc Drop vs Baseline (%)": round(drop * 100, 2) if key != "baseline" else "---"})

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

# Ranked contribution table (ablations only, sorted by drop)
abl_rows = [r for r in rows if r["Model"] != "Baseline (ConvNeXt-Tiny)"]
df_ranked = pd.DataFrame(abl_rows).sort_values("Acc Drop vs Baseline (%)", ascending=False)
print("\nRanked contribution (largest drop = most important component):")
print(df_ranked.to_string(index=False))

## P3 — 6) Loss Curve Plot (Part B)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, history in results_p3.items():
    label = label_map.get(name, name)
    axes[0].plot(range(1, EPOCHS_P3 + 1), history["train_loss"], marker="o", label=label)
    axes[1].plot(range(1, EPOCHS_P3 + 1), history["val_acc"],   marker="o", label=label)

axes[0].set_title("Training Loss (all 6 models)")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=8); axes[0].grid(True)

axes[1].set_title("Validation Accuracy (all 6 models)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend(fontsize=8); axes[1].grid(True)

plt.suptitle(f"PatchCamelyon Ablation Study (seed={SEED})", fontsize=13)
plt.tight_layout()
plt.savefig("p3_ablation_curves.png", dpi=150)
plt.show()

## P3 — 7) Written Analysis (Part B) — fill in below

> **Q: Which single design choice contributes most to ConvNeXt's performance on PatchCamelyon?**

*Your answer here.*

> **Q: Does your ranking agree with the original paper's ablation on ImageNet? Why / why not?** (3–5 sentences)

*Your answer here.*

> **Q: Which ablations slow convergence vs. reduce final accuracy? Are these correlated?**

*Your answer here.*

## P3 — 8) Architectural Proposal (Part C)

> **Proposed modification:**

*Name the modification here.*

> **Justification using your ablation numbers:**

*Your justification here (reference specific accuracy drops from your table).*

> **Estimated GPU cost impact:**

*Increase / decrease and why.*

---
# PROBLEM 4: Anchor-Free vs. Anchor-Based — FCOS vs. RetinaNet Showdown (25 pts)

We fine-tune two detection models on **NWPU VHR-10** aerial imagery and compare  
anchor-free (FCOS) vs. anchor-based (RetinaNet) design philosophy.

## P4 — 1) Dataset: NWPU VHR-10

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# NWPU VHR-10 dataset loader (COCO-format JSON expected at ROOT_P4/annotations/)
# Alternatively, torchgeo.datasets.VHR10 can be used if torchgeo ≥ 0.5:
#   from torchgeo.datasets import VHR10
# We provide a fallback COCO loader below.
# ─────────────────────────────────────────────────────────────────────────────
import os, json
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision import tv_tensors

ROOT_P4 = "./data_vhr10"   # place NWPU VHR-10 images + COCO annotation JSON here
                            # or set COLAB_DRIVE_PATH below and mount Google Drive

VHR10_CLASSES = [
    "__background__",
    "airplane", "ship", "storage_tank", "baseball_diamond",
    "tennis_court", "basketball_court", "ground_track_field",
    "harbor", "bridge", "vehicle",
]

class VHR10COCODataset(Dataset):
    """
    Minimal COCO-format detection dataset for NWPU VHR-10.
    Expects: ROOT_P4/images/*.jpg  and  ROOT_P4/annotations/vhr10.json
    """
    def __init__(self, root, split_indices, transforms=None):
        ann_path   = os.path.join(root, "annotations", "vhr10.json")
        with open(ann_path) as f:
            coco = json.load(f)

        self.root       = root
        self.transforms = transforms
        self.imgs       = [coco["images"][i]      for i in split_indices]
        img_ids         = {img["id"] for img in self.imgs}
        self.anns_by_id = {img["id"]: [] for img in self.imgs}
        for ann in coco["annotations"]:
            if ann["image_id"] in img_ids:
                self.anns_by_id[ann["image_id"]].append(ann)

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_meta = self.imgs[idx]
        img_path = os.path.join(self.root, "images", img_meta["file_name"])
        image    = Image.open(img_path).convert("RGB")
        W, H     = image.size

        anns     = self.anns_by_id[img_meta["id"]]
        boxes    = []
        labels   = []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            boxes.append([x, y, x + w, y + h])
            labels.append(ann["category_id"])

        boxes  = torch.tensor(boxes,  dtype=torch.float32) if boxes  else torch.zeros((0, 4))
        labels = torch.tensor(labels, dtype=torch.int64)   if labels else torch.zeros((0,), dtype=torch.int64)

        # Wrap with tv_tensors for transforms v2
        image  = tv_tensors.Image(image)
        boxes  = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(H, W))

        target = {"boxes": boxes, "labels": labels,
                  "image_id": torch.tensor([img_meta["id"]])}

        if self.transforms:
            # YOUR CODE HERE (Part A): apply self.transforms to image and target
            # Hint: transforms.v2 pipelines accept (image, target) as a tuple
            pass

        return image, target


print("✅ VHR10COCODataset class defined.")

In [ ]:
# ─── Auto-download NWPU VHR-10 via torchgeo & convert to COCO format ────────
# Only the 650 "positive" images (which contain annotated objects) are used.
# torchgeo downloads ~170 MB; conversion copies images + writes a COCO JSON.
import os, json, shutil, gc

ann_out_path = os.path.join(ROOT_P4, "annotations", "vhr10.json")
img_out_dir  = os.path.join(ROOT_P4, "images")

if os.path.exists(ann_out_path) and os.path.isdir(img_out_dir) and len(os.listdir(img_out_dir)) > 0:
    with open(ann_out_path) as _f:
        _meta = json.load(_f)
    print(f"✅ VHR-10 COCO data already prepared — {len(_meta['images'])} images, "
          f"{len(_meta['annotations'])} annotations. Skipping download.")
    del _meta
else:
    print("⬇️  Downloading NWPU VHR-10 via torchgeo (≈170 MB) ...")
    from torchgeo.datasets import VHR10

    _tgeo_root = os.path.join(ROOT_P4, "_torchgeo")
    _ds = VHR10(root=_tgeo_root, split="positive", download=True)

    _tgeo_base = os.path.join(_tgeo_root, "NWPU VHR-10 dataset")
    _tgeo_ann  = os.path.join(_tgeo_base, "annotations.json")
    _tgeo_imgs = os.path.join(_tgeo_base, "positive image set")

    with open(_tgeo_ann) as f:
        coco_data = json.load(f)

    os.makedirs(img_out_dir, exist_ok=True)
    os.makedirs(os.path.dirname(ann_out_path), exist_ok=True)

    copied = 0
    for img_entry in coco_data["images"]:
        src_name  = img_entry["file_name"]
        base_name = os.path.basename(src_name)
        src_path  = os.path.join(_tgeo_imgs, base_name)
        if not os.path.exists(src_path):
            src_path = os.path.join(_tgeo_base, src_name)
        dst_path = os.path.join(img_out_dir, base_name)
        if not os.path.exists(dst_path):
            shutil.copy2(src_path, dst_path)
            copied += 1
        img_entry["file_name"] = base_name

    with open(ann_out_path, "w") as f:
        json.dump(coco_data, f)

    print(f"✅ VHR-10 prepared: {len(coco_data['images'])} images ({copied} copied), "
          f"{len(coco_data['annotations'])} annotations")
    del _ds, coco_data
    gc.collect()

## P4 — 2) Transforms v2 Pipeline (Part A)

In [ ]:
from torchvision.transforms import v2

IMG_SIZE_P4 = 512

# YOUR CODE HERE (Part A-a): define train_tfm_p4 and val_tfm_p4
# train_tfm_p4 must include: Resize to IMG_SIZE_P4, RandomHorizontalFlip(p=0.5),
#                            ColorJitter(brightness=0.2, contrast=0.2), ToImage(), ToDtype
# val_tfm_p4   must include: Resize to IMG_SIZE_P4, ToImage(), ToDtype
# Use v2.Compose([...]) with v2 transforms only.

train_tfm_p4 = None   # replace with your v2.Compose pipeline
val_tfm_p4   = None   # replace with your v2.Compose pipeline


# ─── Build dataset splits with your seed ─────────────────────────────────────
ann_path_p4 = os.path.join(ROOT_P4, "annotations", "vhr10.json")
if os.path.exists(ann_path_p4):
    with open(ann_path_p4) as _f:
        N_IMAGES = len(json.load(_f)["images"])
    print(f"VHR-10 annotation file found — {N_IMAGES} positive images")
else:
    N_IMAGES = 650
    print(f"⚠️  VHR-10 annotations not found at {ann_path_p4} — using N_IMAGES={N_IMAGES}")

all_idx  = np.arange(N_IMAGES)
rng2     = np.random.default_rng(SEED)
rng2.shuffle(all_idx)
split    = int(0.7 * N_IMAGES)
train_idx, val_idx = all_idx[:split], all_idx[split:]

if os.path.exists(ann_path_p4):
    train_det = VHR10COCODataset(ROOT_P4, train_idx, transforms=train_tfm_p4)
    val_det   = VHR10COCODataset(ROOT_P4, val_idx,   transforms=val_tfm_p4)
    print("✅ Using real NWPU VHR-10 COCO annotations")
else:
    print(f"⚠️ VHR-10 annotation file not found at: {ann_path_p4}")
    print("⚠️ Falling back to synthetic detection dataset so notebook remains runnable.")

    class SyntheticDetectionDataset(Dataset):
        def __init__(self, n_images, transforms=None, img_size=512, num_classes=10, seed=0):
            self.n_images = n_images
            self.transforms = transforms
            self.img_size = img_size
            self.num_classes = num_classes
            self.seed = int(seed)

        def __len__(self):
            return self.n_images

        def __getitem__(self, idx):
            rng = np.random.default_rng(self.seed + idx)
            h = w = self.img_size
            image = tv_tensors.Image(torch.randint(0, 256, (3, h, w), dtype=torch.uint8))

            n_boxes = int(rng.integers(1, 6))
            boxes = []
            labels = []
            for _ in range(n_boxes):
                bw = int(rng.integers(24, max(25, w // 3)))
                bh = int(rng.integers(24, max(25, h // 3)))
                x1 = int(rng.integers(0, max(1, w - bw)))
                y1 = int(rng.integers(0, max(1, h - bh)))
                x2 = x1 + bw
                y2 = y1 + bh
                boxes.append([x1, y1, x2, y2])
                labels.append(int(rng.integers(1, self.num_classes + 1)))

            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            boxes = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(h, w))

            target = {
                "boxes": boxes,
                "labels": labels,
                "image_id": torch.tensor([idx], dtype=torch.int64),
            }

            if self.transforms:
                image, target = self.transforms(image, target)
            return image, target

    train_det = SyntheticDetectionDataset(len(train_idx), transforms=train_tfm_p4,
                                          img_size=IMG_SIZE_P4, num_classes=len(VHR10_CLASSES)-1,
                                          seed=SEED)
    val_det   = SyntheticDetectionDataset(len(val_idx), transforms=val_tfm_p4,
                                          img_size=IMG_SIZE_P4, num_classes=len(VHR10_CLASSES)-1,
                                          seed=SEED + 100000)

def collate_detection(batch):
    """Detection models expect List[Tensor] inputs, not a stacked batch."""
    images, targets = zip(*batch)
    return list(images), list(targets)

train_loader_p4 = DataLoader(train_det, batch_size=4, shuffle=True,  num_workers=0,
                             collate_fn=collate_detection, pin_memory=False)
val_loader_p4   = DataLoader(val_det,   batch_size=4, shuffle=False, num_workers=0,
                             collate_fn=collate_detection, pin_memory=False)

print(f"Train: {len(train_det)}  Val: {len(val_det)}")

### Part A — Written Explanation

> **Q: Why is `transforms.v2` with `tv_tensors.BoundingBoxes` necessary for detection augmentation?  
> What happens to box coordinates under a horizontal flip, and how does tv_tensors handle it automatically?** (2–3 sentences)

*Your answer here.*

## P4 — 3) Fine-Tune FCOS and RetinaNet (Part B)

In [ ]:
from torchvision.models.detection import (
    fcos_resnet50_fpn,    FCOS_ResNet50_FPN_Weights,
    retinanet_resnet50_fpn, RetinaNet_ResNet50_FPN_Weights,
)
from torchvision.models.detection.fcos import FCOSClassificationHead
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

NUM_CLASSES_P4 = len(VHR10_CLASSES)   # 11 (including background)
EPOCHS_P4      = 5

def make_fcos():
    # Load COCO-pretrained FCOS first (fixed 91 classes), then swap cls head.
    model = fcos_resnet50_fpn(weights=FCOS_ResNet50_FPN_Weights.COCO_V1)
    old_head = model.head.classification_head
    in_channels = old_head.cls_logits.in_channels
    num_anchors = old_head.num_anchors
    model.head.classification_head = FCOSClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=NUM_CLASSES_P4,
    )
    return model.to(device)

def make_retinanet():
    # Same pattern for RetinaNet: keep COCO backbone+features, replace cls head.
    model = retinanet_resnet50_fpn(weights=RetinaNet_ResNet50_FPN_Weights.COCO_V1)
    old_head = model.head.classification_head
    in_channels = old_head.cls_logits.in_channels
    num_anchors = old_head.num_anchors
    model.head.classification_head = RetinaNetClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=NUM_CLASSES_P4,
    )
    return model.to(device)


# ─── GAP 1: Optimizer with Differential Learning Rates ───────────────────────
def make_optimizer_p4(model):
    """
    YOUR CODE HERE (Part B — Gap 1):
    Create an SGD optimizer with:
      - backbone parameters: lr = 1e-4
      - all other parameters (head): lr = 1e-2
      - momentum = 0.9, weight_decay = 1e-4

    Hint: use model.backbone.parameters() for backbone params.
    Return the optimizer.
    """
    # YOUR CODE HERE
    raise NotImplementedError


print("✅ Detection model factories ready.")

In [ ]:
from torchvision.ops import box_iou

# ─── GAP 3 helper: post-processing (NMS + score threshold) ───────────────────
def postprocess_predictions(raw_preds, score_thresh=0.3, nms_iou=0.5):
    """
    YOUR CODE HERE (Part B — Gap 3):
    Given raw_preds (list of dicts with 'boxes', 'scores', 'labels'),
    for each image:
      1. Filter predictions with score < score_thresh.
      2. Apply NMS (torchvision.ops.nms) with iou_threshold=nms_iou.
    Return filtered list of dicts.
    """
    from torchvision.ops import nms
    filtered = []
    for pred in raw_preds:
        # YOUR CODE HERE
        pass
    return filtered


# ─── mAP computation (simple IoU-based, @0.5) ─────────────────────────────────
def compute_map_50(predictions, targets, iou_thresh=0.5, num_classes=NUM_CLASSES_P4):
    """Simple per-class AP @ IoU=0.5, then mean over classes."""
    from collections import defaultdict

    tp_fp   = defaultdict(list)   # per class: list of (score, is_tp)
    n_gt    = defaultdict(int)    # per class: count of GT boxes

    for pred, tgt in zip(predictions, targets):
        gt_boxes  = tgt["boxes"].cpu()
        gt_labels = tgt["labels"].cpu()

        if len(pred["boxes"]) == 0:
            for lbl in gt_labels:
                n_gt[lbl.item()] += 1
            continue

        pred_boxes  = pred["boxes"].cpu()
        pred_scores = pred["scores"].cpu()
        pred_labels = pred["labels"].cpu()

        for lbl in gt_labels:
            n_gt[lbl.item()] += 1

        matched_gt = set()
        for i in pred_scores.argsort(descending=True):
            cls = pred_labels[i].item()
            gt_mask = (gt_labels == cls).nonzero(as_tuple=True)[0]
            if len(gt_mask) == 0:
                tp_fp[cls].append((pred_scores[i].item(), 0))
                continue
            ious = box_iou(pred_boxes[i:i+1], gt_boxes[gt_mask])[0]
            best = ious.argmax().item()
            g_idx = gt_mask[best].item()
            if ious[best] >= iou_thresh and g_idx not in matched_gt:
                tp_fp[cls].append((pred_scores[i].item(), 1))
                matched_gt.add(g_idx)
            else:
                tp_fp[cls].append((pred_scores[i].item(), 0))

    aps = {}
    for cls in range(1, num_classes):
        if n_gt[cls] == 0:
            continue

        pairs = sorted(tp_fp[cls], key=lambda x: -x[0])
        if len(pairs) == 0:
            aps[cls] = 0.0
            continue

        tp_cum = np.cumsum([p[1] for p in pairs])
        fp_cum = np.cumsum([1 - p[1] for p in pairs])
        prec   = tp_cum / (tp_cum + fp_cum + 1e-9)
        rec    = tp_cum / (n_gt[cls] + 1e-9)

        ap_vals = []
        for t in np.linspace(0, 1, 11):
            valid = prec[rec >= t]
            ap_vals.append(float(valid.max()) if valid.size > 0 else 0.0)
        aps[cls] = float(np.mean(ap_vals))

    return aps, float(np.mean(list(aps.values()))) if aps else 0.0


print("✅ mAP utilities ready.")

In [ ]:
def train_detector(model, model_name):
    """Full training loop for one detector."""
    set_seed(SEED)
    optimizer = make_optimizer_p4(model)
    history   = {"train_loss": [], "val_map": []}

    for epoch in range(1, EPOCHS_P4 + 1):
        # ── Training ─────────────────────────────────────────────────────────
        model.train()
        epoch_loss = 0.0

        for images, targets in train_loader_p4:
            images  = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # GAP 2: Loss forward pass ─────────────────────────────────────────
            # YOUR CODE HERE (Part B — Gap 2):
            # 1. Zero gradients.
            # 2. Call model(images, targets) — returns a loss dict in training mode.
            # 3. Sum all values in the loss dict to get total_loss.
            # 4. Backprop and optimizer step.
            # 5. Add total_loss.item() * len(images) to epoch_loss.
            raise NotImplementedError   # remove this line when you fill in the gap

        train_loss = epoch_loss / len(train_loader_p4.dataset)

        # ── Validation ───────────────────────────────────────────────────────
        model.eval()
        all_preds, all_tgts = [], []
        with torch.no_grad():
            for images, targets in val_loader_p4:
                images = [img.to(device) for img in images]
                raw    = model(images)   # eval mode → list of prediction dicts
                preds  = postprocess_predictions(raw)
                all_preds.extend(preds)
                all_tgts.extend(targets)

        _, val_map = compute_map_50(all_preds, all_tgts)

        history["train_loss"].append(train_loss)
        history["val_map"].append(val_map)
        print(f"[{model_name}] Epoch {epoch:02d} | loss={train_loss:.4f} | mAP@0.5={val_map:.4f}")

    return model, history


print("✅ Training loop ready — remember to fill in Gap 2 before running!")

In [ ]:
# Run both detectors
results_p4 = {}

print("Training FCOS...")
fcos_model, results_p4["fcos"]     = train_detector(make_fcos(),      "FCOS")

print("\nTraining RetinaNet...")
rnet_model, results_p4["retinanet"] = train_detector(make_retinanet(), "RetinaNet")

## P4 — 4) Training Curves (Part B)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs_x = range(1, EPOCHS_P4 + 1)

for name, color in [("fcos", "steelblue"), ("retinanet", "darkorange")]:
    axes[0].plot(epochs_x, results_p4[name]["train_loss"], marker="o", color=color, label=name.upper())
    axes[1].plot(epochs_x, results_p4[name]["val_map"],   marker="o", color=color, label=name.upper())

axes[0].set_title("Training Loss");    axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True)
axes[1].set_title("Validation mAP@0.5"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True)
plt.suptitle(f"FCOS vs RetinaNet on NWPU VHR-10 (seed={SEED})", fontsize=13)
plt.tight_layout()
plt.savefig("p4_detection_curves.png", dpi=150)
plt.show()

## P4 — 5) Per-Class AP Table (Part C)

In [ ]:
def get_per_class_ap(model):
    model.eval()
    all_preds, all_tgts = [], []
    with torch.no_grad():
        for images, targets in val_loader_p4:
            images = [img.to(device) for img in images]
            raw    = model(images)
            preds  = postprocess_predictions(raw)
            all_preds.extend(preds)
            all_tgts.extend(targets)
    aps, mean_ap = compute_map_50(all_preds, all_tgts)
    return aps, mean_ap

fcos_aps, fcos_map = get_per_class_ap(fcos_model)
rnet_aps, rnet_map = get_per_class_ap(rnet_model)

rows = []
for cls_id in range(1, NUM_CLASSES_P4):
    rows.append({
        "Class"         : VHR10_CLASSES[cls_id],
        "FCOS AP@0.5"   : round(fcos_aps.get(cls_id, 0.0), 4),
        "RetinaNet AP@0.5": round(rnet_aps.get(cls_id, 0.0), 4),
        "Gap (FCOS-Rnet)": round(fcos_aps.get(cls_id, 0.0) - rnet_aps.get(cls_id, 0.0), 4),
    })

rows.append({"Class": "Overall mAP",
             "FCOS AP@0.5": round(fcos_map, 4),
             "RetinaNet AP@0.5": round(rnet_map, 4),
             "Gap (FCOS-Rnet)": round(fcos_map - rnet_map, 4)})

df_ap = pd.DataFrame(rows)
print(df_ap.to_string(index=False))

## P4 — 6) 2×5 Detection Visualization Grid (Part C)

In [ ]:
import matplotlib.patches as patches

CLASS_COLORS = plt.cm.get_cmap("tab10", NUM_CLASSES_P4)

def draw_detections(ax, image_tensor, preds, title, score_thresh=0.3):
    img_np = image_tensor.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    ax.imshow(img_np)
    for box, label, score in zip(preds["boxes"], preds["labels"], preds["scores"]):
        if score < score_thresh:
            continue
        x1, y1, x2, y2 = box.cpu().tolist()
        cls_name = VHR10_CLASSES[label.item()]
        color    = CLASS_COLORS(label.item())
        rect     = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                      linewidth=1.5, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f"{cls_name} {score:.2f}",
                color=color, fontsize=6, weight="bold")
    ax.set_title(title, fontsize=8)
    ax.axis("off")


# Select 5 validation images for visualization
vis_indices = list(range(min(5, len(val_det))))
fig, axes   = plt.subplots(2, 5, figsize=(20, 8))

fcos_model.eval(); rnet_model.eval()
with torch.no_grad():
    for col, img_idx in enumerate(vis_indices):
        img_tensor, tgt = val_det[img_idx]
        inp = [img_tensor.to(device)]

        fcos_raw  = postprocess_predictions(fcos_model(inp))[0]
        rnet_raw  = postprocess_predictions(rnet_model(inp))[0]

        draw_detections(axes[0][col], img_tensor, fcos_raw,  f"FCOS  | img {img_idx}")
        draw_detections(axes[1][col], img_tensor, rnet_raw,  f"RetNet| img {img_idx}")

plt.suptitle(f"Detection Visualizations (seed={SEED})", fontsize=13)
plt.tight_layout()
plt.savefig("p4_detection_grid.png", dpi=150)
plt.show()

### Part C — Written Analysis

> **Q: Which two classes show the largest AP gap between FCOS and RetinaNet?  
> What visual characteristics (size, aspect ratio, density) explain the gap?**

*Your answer here.*

## P4 — 7) Anchor Analysis (Part D)

In [ ]:
# ─── Part D-a: Visualize RetinaNet anchors at 3 FPN levels ────────────────────
import torchvision.transforms.functional as TF

def visualize_retinanet_anchors(model, img_tensor, fpn_levels=("0", "1", "2"), max_anchors=50):
    """
    Extract and draw RetinaNet anchor boxes at the specified FPN levels
    (P3=level 0, P4=level 1, P5=level 2 in torchvision indexing).
    """
    model.to(device).eval()
    images = [img_tensor.to(device)]

    # AnchorGenerator expects an ImageList (not a plain Python list).
    with torch.no_grad(), torch.amp.autocast(device_type='cuda', enabled=torch.cuda.is_available()):
        image_list, _ = model.transform(images, None)
        features = model.backbone(image_list.tensors)
        if isinstance(features, torch.Tensor):
            features = {"0": features}
        feature_maps = list(features.values())
        anchors = model.anchor_generator(image_list, feature_maps)

    anchor_boxes = anchors[0].cpu()
    total = len(anchor_boxes)
    print(f"Total anchors for a {IMG_SIZE_P4}x{IMG_SIZE_P4} image: {total}")

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    ax.imshow(img_np)

    num_anchors_per_loc = model.anchor_generator.num_anchors_per_location()[0]
    grid_sizes = [fm.shape[-2:] for fm in feature_maps]
    level_counts = [h * w * num_anchors_per_loc for (h, w) in grid_sizes]

    colors = ["red", "lime", "cyan", "yellow", "magenta"]
    offset = 0
    for lvl, n in enumerate(level_counts):
        lvl_anchors = anchor_boxes[offset: offset + n]
        offset += n
        if len(lvl_anchors) == 0:
            continue
        sample_idx = np.random.choice(len(lvl_anchors), size=min(max_anchors, len(lvl_anchors)), replace=False)
        color = colors[lvl % len(colors)]
        for idx in sample_idx:
            x1, y1, x2, y2 = lvl_anchors[idx].tolist()
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                     linewidth=0.8, edgecolor=color, facecolor="none", alpha=0.6)
            ax.add_patch(rect)
        ax.plot([], [], color=color, label=f"P{lvl + 3} (~{n} anchors)")

    ax.legend(fontsize=9)
    ax.set_title("RetinaNet Anchors at FPN Levels")
    ax.axis("off")
    plt.tight_layout()
    plt.savefig("p4_anchor_visualization.png", dpi=150)
    plt.show()
    return total


# Run on one validation image
sample_img, _ = val_det[0]
total_anchors = visualize_retinanet_anchors(rnet_model, sample_img)

### Part D — Written Analysis

> **D-b: Explain how FCOS center-sampling (radius=1.5) assigns ground-truth objects to feature map locations, and why this is advantageous for objects with unusual aspect ratios such as ships and bridges.** (2–3 sentences)

*Your answer here.*

> **D-c: Using your specific AP numbers from Part C, argue which detection paradigm is better suited for aerial object detection on VHR-10 and why. Reference at least two specific classes.** (3–5 sentences)

*Your answer here.*

In [ ]:
# Final compute diary stamp — run this at the END of each session
from datetime import datetime
import json, os
stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
with open("diary/compute_diary.txt", "a") as f:
    f.write(json.dumps({"event": "session_end", "when": stamp}) + "\n")
print("⏱️ Session end logged:", stamp)